# Notebook 02: Bilingual Query Understanding

## Overview

Before searching, the system must understand the user's question:
- **What language** is it in? (Arabic or English)
- **What intent** does it have? (treatment, symptoms, follow-up, etc.)
- **How to reformulate** it for better retrieval? (domain context, multiple queries)

This happens entirely locally -- no API calls, no LLM needed.

```
User question  -->  Normalize  -->  Classify Intent  -->  Expand Queries
```

**Source file:** `src/rag_app/retrieval/query_understanding.py`

## 1. Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag_app import config
from src.rag_app.retrieval.query_understanding import (
    INTENTS,
    normalize_question,
    add_domain_context,
    understand_question,
    keyword_score,
)
from sentence_transformers import SentenceTransformer

## 2. The 7 Intent Categories

Each intent has:
- A **description** used for semantic similarity fallback
- **Cue words** in both Arabic and English for fast matching

In [ ]:
for intent_name, intent_data in INTENTS.items():
    print(f"{intent_name}:")
    print(f"  Description: {intent_data['description']}")
    print(f"  Arabic cues: {[c for c in intent_data['cues'] if any(ord(ch) > 0x0600 for ch in c)]}")
    print(f"  English cues: {[c for c in intent_data['cues'] if all(ord(ch) < 0x0600 for ch in c)]}")
    print()

## 3. Arabic Normalization

Arabic text has many valid spellings. Normalization ensures consistent matching:
- Remove diacritics (tashkeel)
- Normalize alef variants (أ, إ, آ --> ا)
- Normalize ya (ى --> ي)
- Remove tatweel (ـ)
- NFC unicode normalization

In [ ]:
# Examples of Arabic normalization
test_cases = [
    "ما هي أعراض سرطان القولون؟",          # with hamza
    "ما هي اعراض سرطان القولون؟",          # without hamza (same meaning)
    "Follow-up after surgery",              # English (no normalization needed)
    "عِلاج السَّرَطان",                      # with diacritics
]

for text in test_cases:
    normalized = normalize_question(text)
    print(f"Original:  '{text}'")
    print(f"Normalized: '{normalized}'")
    print()

## 4. Intent Classification: Cue Matching

The primary method is **cue matching** -- if the question contains known cue words, we know the intent immediately with confidence 1.0.

In [ ]:
# Cue-based classification examples
cue_examples = [
    ("ما هي أعراض سرطان القولون؟", "symptoms_referral"),
    ("عندي سرطان اعمل اي", "newly_diagnosed_information"),
    ("متابعة بعد الجراحة", "follow_up"),
    ("علاج كيماوي", "treatment"),
    ("آثار جانبية", "side_effects"),
    ("جراحة استئصال", "surgery"),
    ("طفرة BRAF", "biomarkers"),
]

for question, expected_intent in cue_examples:
    normalized = normalize_question(question)
    scores = {
        name: sum(normalize_question(cue) in normalized for cue in data["cues"])
        for name, data in INTENTS.items()
    }
    best = max(scores, key=scores.get)
    match = "OK" if best == expected_intent else "MISMATCH"
    print(f"'{question}'")
    print(f"  Detected: {best} (confidence: 1.0) [{match}]")
    print()

## 5. Intent Classification: Semantic Fallback

When no cue words match, we use the embedding model to compute cosine similarity between the question and each intent description.

In [ ]:
model = SentenceTransformer(
    config.EMBEDDING_MODEL,
    local_files_only=config.EMBEDDING_LOCAL_FILES_ONLY,
)

# A question without obvious cue words
question = "What tests should be done to detect the disease early?"
normalized = normalize_question(question)

# Check if any cues match
scores = {
    name: sum(normalize_question(cue) in normalized for cue in data["cues"])
    for name, data in INTENTS.items()
}
cue_intent = max(scores, key=scores.get)
print(f"Cue scores: {scores}")
print(f"Best cue match: {cue_intent} (score: {scores[cue_intent]})")

if scores[cue_intent] == 0:
    print("\nNo cues matched. Using semantic similarity fallback...")
    texts = [f"query: {normalized}"] + [f"passage: {data['description']}" for data in INTENTS.values()]
    vectors = model.encode(texts, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    similarities = [float(vectors[0] @ vector) for vector in vectors[1:]]
    
    for name, sim in zip(INTENTS.keys(), similarities):
        print(f"  {name}: {sim:.4f}")
    
    best_idx = max(range(len(similarities)), key=similarities.__getitem__)
    best_intent = list(INTENTS.keys())[best_idx]
    print(f"\nSemantic intent: {best_intent} (similarity: {similarities[best_idx]:.4f})")

## 6. Domain Context Expansion

Short questions like "what are the symptoms?" are ambiguous. Since this is a **specialist app** (colorectal cancer only), we prepend domain context to improve retrieval.

In [ ]:
short_questions = [
    "ما هي الأعراض؟",                    # Arabic, no cancer mention
    "What are the symptoms?",              # English, no cancer mention
    "سرطان القولون والمستقيم",              # Already has domain context
    "colorectal cancer treatment",         # Already has domain context
]

for q in short_questions:
    expanded = add_domain_context(q)
    changed = " --> EXPANDED" if expanded != q else " (already has context)"
    print(f"Original: '{q}'")
    print(f"Expanded: '{expanded}'{changed}")
    print()

## 7. Full Understanding Pipeline

The `understand_question()` function combines everything:
1. Normalize the question
2. Classify intent (cue match or semantic fallback)
3. Expand with domain context
4. Generate multiple query formulations for robust retrieval

In [ ]:
# Full understanding for an Arabic question
question = "ايه العلاج بعد الجراحة؟"
result = understand_question(question, model)

print(f"Original: {result['original']}")
print(f"Normalized: {result['normalized']}")
print(f"Domain question: {result['domain_question']}")
print(f"Intent: {result['intent']} (confidence: {result['confidence']:.2f})")
print(f"\nGenerated queries for retrieval:")
for i, q in enumerate(result['queries'], 1):
    print(f"  {i}. {q}")

In [ ]:
# Full understanding for an English question
question = "What follow-up is needed after curative surgery?"
result = understand_question(question, model)

print(f"Original: {result['original']}")
print(f"Normalized: {result['normalized']}")
print(f"Intent: {result['intent']} (confidence: {result['confidence']:.2f})")
print(f"\nGenerated queries for retrieval:")
for i, q in enumerate(result['queries'], 1):
    print(f"  {i}. {q}")

## 8. Keyword Scoring

A simple lexical overlap score is used alongside vector similarity during reranking.

In [ ]:
query = "What follow-up is needed after surgery?"
documents = [
    "Follow-up for detection of local recurrence and distant metastases.",
    "Surgical technique for people with rectal cancer.",
    "BRAF V600E mutation-positive disease treatment options.",
]

print(f"Query: '{query}'")
print(f"\nKeyword scores:")
for doc in documents:
    score = keyword_score(doc, query)
    print(f"  {score:.3f} - {doc[:60]}...")

## Summary

| Step | Method | Output |
|------|--------|--------|
| Normalization | Unicode NFC, Arabic diacritics removal, alef/ya normalization | Consistent text for matching |
| Intent classification | Cue matching (confidence=1.0) or cosine similarity (threshold=0.73) | Intent category + confidence |
| Domain expansion | Prepend colorectal cancer context to short questions | Better retrieval for ambiguous queries |
| Query formulation | Original + normalized + domain-expanded + intent description | Multiple queries for robust retrieval |
| Keyword scoring | Lexical overlap (words > 2 chars) | Numeric boost for reranking |

**Key insight:** By generating multiple query formulations and using both semantic + lexical signals, the system handles the ambiguity of short bilingual questions without any LLM call.